PROJECT 1 : LIVE MARKET DATA PIPELINE & DASHBOARD

In [18]:
## installing the yfinance package that fetches real stock from yahoo finance

!pip install yfinance pandas

In [19]:
## asking for price data of last five days from y finance

import yfinance as yf

apple = yf.Ticker("AAPL")
data = apple.history(period ="5d")

data

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-06-29 00:00:00-04:00,286.730011,288.369995,279.850006,281.739990,66427000,0.0,0.0
2026-06-30 00:00:00-04:00,281.170013,289.940002,280.700012,289.359985,65100200,0.0,0.0
2026-07-01 00:00:00-04:00,293.440002,296.589996,289.200012,294.380005,50164200,0.0,0.0
2026-07-02 00:00:00-04:00,294.119995,309.420013,293.679993,308.630005,75352800,0.0,0.0
2026-07-06 00:00:00-04:00,307.359985,314.200012,307.010010,312.660004,49130304,0.0,0.0


In [20]:
## Defining a list of stickers and looping through them to collect each ones data
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN","TLSA"]
all_data = {}
for ticker in tickers:
    stock = yf.Ticker(ticker)
    all_data[ticker] = stock.history(period = "5d")
## We just automated fetching for different companies at once, instead of doing it manually five times   
all_data["AAPL"]

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-06-29 00:00:00-04:00,286.730011,288.369995,279.850006,281.739990,66427000,0.0,0.0
2026-06-30 00:00:00-04:00,281.170013,289.940002,280.700012,289.359985,65100200,0.0,0.0
2026-07-01 00:00:00-04:00,293.440002,296.589996,289.200012,294.380005,50164200,0.0,0.0
2026-07-02 00:00:00-04:00,294.119995,309.420013,293.679993,308.630005,75352800,0.0,0.0
2026-07-06 00:00:00-04:00,307.359985,314.200012,307.010010,312.660004,49130304,0.0,0.0


In [21]:
all_data["GOOGL"]

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-06-29 00:00:00-04:00,341.829987,354.350006,340.670013,353.649994,34213900,0.0,0.0
2026-06-30 00:00:00-04:00,353.869995,358.619995,350.399994,357.369995,35237200,0.0,0.0
2026-07-01 00:00:00-04:00,358.369995,362.970001,356.429993,361.209991,26736400,0.0,0.0
2026-07-02 00:00:00-04:00,359.480011,364.209991,353.420013,359.910004,25974800,0.0,0.0
2026-07-06 00:00:00-04:00,361.549988,367.929993,357.380493,366.459991,26296118,0.0,0.0


In [22]:
## Storing as a csv file
import pandas as pd

all_data["AAPL"].to_csv("AAPL_prices.csv")

In [23]:
## using the loop to store the five different stickers as csv files

for ticker in tickers:
    print(f"Saving{ticker}...")
    all_data[ticker].to_csv(f"{ticker}_prices.csv")
print("Done")

## This creates a csv file that does not append in case of a rerun, but rather overwrites, to solve this issue, we use databases (SQL)

SavingAAPL...
SavingMSFT...
SavingGOOGL...
SavingAMZN...
SavingTLSA...
Done


In [24]:
## But python does not talk to SQL directly, so we need a translator package

!pip install sqlalchemy pyodbc

In [25]:
## creating a connection object btw python and sql
from sqlalchemy import create_engine

server = 'localhost'
database = 'finance_pipeline'

connection_string = f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
## driver is the specific software translator, trusted_connection tells it to use Window Authentication
engine = create_engine(connection_string)

In [26]:
## Testing if the connection went through
import pandas as pd
test = pd.read_sql("SELECT 1 AS test_column", engine)

In [27]:
test
## since this printed, it shows that the connection went through and that python successfully talked to sql

,test_column
0,1


In [28]:
## saving our real data into sql
aapl_df = all_data["AAPL"].reset_index()
aapl_df["Date"] = aapl_df["Date"].dt.tz_localize(None) ## this is done to scrap out the timezone, to enable sql read the data

aapl_df.to_sql("AAPL_prices", engine, if_exists="append", index=False)
## the append is there to cancel theproblem of overwriting, it adds the new data anytime we rerun the code#
## On the other hand, this will produce duplicates as every rerun appends new rows whether already existing or not
## for this cause, the pipeline needs the rule **only insert a row if it is not already there**

5

In [29]:
## pulling out a wider range of data (like past one month)
wider_data = apple.history(period="1mo")
wider_data = wider_data.reset_index()
wider_data["Date"] = wider_data["Date"].dt.tz_localize(None)
wider_data.to_sql("AAPL_prices", engine, if_exists="replace", index=False)

19

In [30]:
## Building the rule
existing = pd.read_sql("SELECT DISTINCT Date FROM AAPL_prices",engine)
existing_dates = existing["Date"].tolist()

new_data = wider_data[~wider_data["Date"].isin(existing_dates)]

## storing it in the database
new_data.to_sql("AAPL_prices", engine, if_exists="append", index=False)

new_data

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits


In [ ]:
## In conclusion, the working funtion for updating sticker is;

def update_ticker(ticker_symbol, table_name):
    stock = yf.Ticker(ticker_symbol)
    new_fetch = stock.history(period="5d").reset_index()
    new_fetch["Date"] = new_fetch["Date"].dt.tz_localize(None)

    try:
        existing = pd.read_sql(f"SELECT DISTINCT Date FROM {table_name}", engine)
        existing_dates = existing["Date"].tolist()
    except:
        existing_dates = []

    new_data = new_fetch[~new_fetch["Date"].isin(existing_dates)]

    if len(new_data) > 0:
        new_data.to_sql(table_name, engine, if_exists="append", index=False)
        print(f"{ticker_symbol}: added {len(new_data)} new rows")
    else:
        print(f"{ticker_symbol}: no new data to add")

update_ticker("AAPL","AAPL_prices")

## History period of 5d is there incase we miss a day, so we can update the missed dates

AAPL: no new data to add


In [34]:
## running the function for all 5 stickers
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

for ticker in tickers:
    table_name = f"{ticker}_prices"
    update_ticker(ticker, table_name)

AAPL: no new data to add
MSFT: added 5 new rows
GOOGL: added 5 new rows
AMZN: added 5 new rows
TSLA: added 5 new rows
